<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms-AI-W6/blob/main/W6D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
from datetime import date
from groq import Groq

# Read the API key from the environment variable used in D1.
api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY is missing. Set it in your terminal first."
    )

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-20b"


def ask_model(messages, temperature=0.1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response


def print_usage(response):
    usage = response.usage
    print(f"Prompt tokens: {usage.prompt_tokens}")
    print(f"Completion tokens: {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")


# =====================================================
# Experiment 1: Temperature
# Summarize the same text 3 times at each temperature.
# =====================================================

text_to_summarize = """
Artificial intelligence is increasingly used in education, healthcare,
transportation, agriculture, and business. In education, AI tools help
students practise skills and receive feedback at their own pace.
Teachers can use these tools to identify learning difficulties, but
they must still check the quality of automated feedback.

In healthcare, machine learning systems can help analyse medical images
and identify patterns in large datasets. These systems may support
professionals, but they should not replace qualified clinical judgment.
In transportation, AI is used in route planning, traffic prediction,
and driver-assistance systems. In agriculture, sensors and predictive
models help farmers monitor soil conditions and manage water use.

Businesses also use AI to organise information, automate repetitive
work, and assist customer service. However, AI systems can make
mistakes, reflect biases in their training data, or perform poorly
in unfamiliar situations. Personal information must be handled carefully,
and important decisions need appropriate oversight.

People should understand the limitations of AI instead of assuming
automated results are always correct. Responsible use requires testing,
monitoring, transparency, privacy protections, and human accountability.
AI can be useful when it supports people and is evaluated carefully.
"""

temperature_results = []

for temperature in (0.1, 0.9):
    for run in range(1, 4):
        print("\n" + "=" * 60)
        print(f"TEMPERATURE: {temperature} | RUN: {run}")
        print("=" * 60)

        response = ask_model(
            [
                {
                    "role": "system",
                    "content": (
                        "Summarize the supplied text in about 80 words. "
                        "Be accurate and do not add unsupported facts."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        "Summarize this text:\n\n" + text_to_summarize
                    ),
                },
            ],
            temperature=temperature,
        )

        answer = response.choices[0].message.content or ""
        usage = response.usage

        print(answer)
        print_usage(response)

        temperature_results.append({
            "temperature": temperature,
            "run": run,
            "answer": answer,
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
        })


# =====================================================
# Experiment 2: Hallucination tests
# =====================================================

questions = [
    (
        "Verifiable fact 1",
        "What is the capital city of Australia?"
    ),
    (
        "Verifiable fact 2",
        "Who wrote Pride and Prejudice?"
    ),
    (
        "Invented book",
        "Give the author, publication year, ISBN, and plot of "
        "The Atlas of Silent Moons by Elian Voss. If you cannot "
        "verify that this book exists, say so."
    ),
    (
        "Invented institution",
        "Describe research published in 2023 by the Institute of "
        "Temporal Botany in Oslo. If you cannot verify that this "
        "institution exists, say so."
    ),
    (
        "Very recent event",
        "What major scientific discovery was announced yesterday? "
        "Give its date and reliable sources. If you cannot verify "
        "a specific event, say you are not sure."
    ),
]

hallucination_results = []

for label, question in questions:
    print("\n" + "=" * 60)
    print(f"HALLUCINATION TEST: {label}")
    print("=" * 60)
    print("Question:", question)

    response = ask_model(
        [
            {
                "role": "system",
                "content": (
                    "Answer carefully. Do not invent facts, sources, "
                    "books, people, or institutions. If you are unsure, "
                    "state your uncertainty clearly."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.1,
    )

    answer = response.choices[0].message.content or ""
    print("\nAnswer:\n", answer)
    print_usage(response)

    hallucination_results.append({
        "label": label,
        "question": question,
        "answer": answer,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    })


# =====================================================
# Experiment 3: Reduce hallucination using a better prompt
# =====================================================

test_question = (
    "Give the author, publication year, ISBN, and plot of "
    "The Atlas of Silent Moons by Elian Voss."
)

print("\n" + "=" * 60)
print("HALLUCINATION REDUCTION")
print("=" * 60)

# Before: no special instruction about uncertainty.
before_response = ask_model(
    [{"role": "user", "content": test_question}],
    temperature=0.1,
)

before = before_response.choices[0].message.content or ""
print("\nBEFORE:\n", before)
print_usage(before_response)

# After: instruct the model not to invent missing information.
after_response = ask_model(
    [
        {
            "role": "system",
            "content": (
                "Do not invent books, authors, ISBNs, dates, or plots. "
                "If you cannot verify that a book exists, explicitly "
                "say: 'I cannot verify that this book exists.' "
                "Distinguish known facts from uncertainty."
            ),
        },
        {"role": "user", "content": test_question},
    ],
    temperature=0.1,
)

after = after_response.choices[0].message.content or ""
print("\nAFTER:\n", after)
print_usage(after_response)


# =====================================================
# Save results to notes.md
# =====================================================

with open("notes.md", "a", encoding="utf-8") as file:
    file.write("\n\n# W6D2: Temperature and Hallucination\n")
    file.write(f"\nExperiment date: {date.today().isoformat()}\n")
    file.write(f"\nModel: `{MODEL}`\n")

    file.write("\n## 1. Temperature experiment\n")
    file.write(
        "\nThe same text was summarized three times at temperature "
        "0.1 and three times at temperature 0.9.\n"
    )
    file.write(
        "\n| Temperature | Run | Prompt tokens | Completion tokens | "
        "Total tokens |\n"
    )
    file.write("|---:|---:|---:|---:|---:|\n")

    for item in temperature_results:
        file.write(
            f"| {item['temperature']} | {item['run']} | "
            f"{item['prompt_tokens']} | "
            f"{item['completion_tokens']} | "
            f"{item['total_tokens']} |\n"
        )

    for item in temperature_results:
        file.write(
            f"\n### Temperature {item['temperature']} "
            f"- Run {item['run']}\n\n"
        )
        file.write(item["answer"] + "\n")

    file.write("\n## 2. Hallucination tests\n")

    for item in hallucination_results:
        file.write(f"\n### {item['label']}\n")
        file.write(f"\n**Question:** {item['question']}\n")
        file.write(f"\n**Answer:** {item['answer']}\n")
        file.write(
            f"\n**Token usage:** prompt={item['prompt_tokens']}, "
            f"completion={item['completion_tokens']}, "
            f"total={item['total_tokens']}\n"
        )
        file.write("\n**Hallucinated?** To be checked.\n")
        file.write(
            "**How I checked:** Verify factual claims using reliable "
            "sources and record the source here.\n"
        )

    file.write("\n## 3. Reducing hallucination\n")
    file.write(f"\n**Question:** {test_question}\n")
    file.write(f"\n**Before improved prompt:** {before}\n")
    file.write(f"\n**After improved prompt:** {after}\n")
    file.write(
        "\n**Comparison:** Check whether the improved prompt reduced "
        "unsupported claims or encouraged honest uncertainty.\n"
    )

    file.write("\n## 4. Can hallucinations be eliminated completely?\n")
    file.write(
        "\nHallucinations cannot be guaranteed to disappear completely. "
        "In a real system, I would retrieve information from reliable "
        "sources, ask the model to express uncertainty, verify important "
        "claims, and use human review for high-impact decisions.\n"
    )

print("\nAll experiments completed.")
print("Results were appended to notes.md.")
print("Review the answers and complete the hallucination evaluations.")